# 介紹
本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [ ]:
import pyautogui as pyg
pyg.sleep(2)
basex=75
basey=320
gap=40

In [ ]:
import pyautogui as pyg
pyg.sleep(2)
basex=65
basey=470
gap=40
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)

In [ ]:
def get_gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [ ]:
def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [ ]:
def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left_c = c - 1
    left_color = get_gem_color(r, left_c, im)
    current_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, left_c):
            return current_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and color_after_swap(nr, nc) == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, left_c) or makes_match(r, c)

In [ ]:
def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [ ]:
def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [ ]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

In [ ]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

In [ ]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

In [ ]:
import time

start_time = time.time()
im = pyg.screenshot()

while True:
    if time.time() - start_time > 60:
        break

    for r in range(8):
        for c in range(8):
            if time.time() - start_time > 60:
                break

            directions = (
                (can_move_left, 0, -1),
                (can_move_right, 0, 1),
                (can_move_up, -1, 0),
                (can_move_down, 1, 0),
            )
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2)
                    pyg.sleep(0.2)
                    im = pyg.screenshot()
                    break

        if time.time() - start_time > 60:
            break